In [ ]:
# 1. Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 2. Install dependencies
%%capture
!pip install unsloth
!pip install --upgrade trl peft accelerate bitsandbytes datasets

In [ ]:
# 3. Check data files
import os
assert os.path.exists('/content/train.jsonl'), 'train.jsonl not found'
assert os.path.exists('/content/val.jsonl'), 'val.jsonl not found'
print('Data files found!')

In [ ]:
# 4. Load model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=2048,
    load_in_4bit=True,
)
print('Model loaded')

In [ ]:
# 5. Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# 6. Load dataset
from datasets import load_dataset
dataset = load_dataset('json', data_files={'train': 'data/train.jsonl', 'validation': 'data/val.jsonl'})
print('Train:', len(dataset['train']), '| Val:', len(dataset['validation']))

In [ ]:
# 7. Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=2048,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        eval_steps=100,
        save_steps=100,
        output_dir='/content/checkpoints',
        optim='adamw_8bit',
        seed=42,
        report_to='none',
    ),
)
trainer.train()

In [ ]:
# 8. Save and download adapter
import shutil
model.save_pretrained('nepallawft-lora')
tokenizer.save_pretrained('nepallawft-lora')
shutil.make_archive('nepallawft-lora', 'zip', 'nepallawft-lora')
files.download('nepallawft-lora.zip')
print('Done')

In [ ]:
# 9. Quick test
FastLanguageModel.for_inference(model)

SYSTEM = 'You are CivicLens, a legal assistant for Nepal\'s laws and governance documents.'

def ask(question):
    prompt = (f'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{SYSTEM}<|eot_id|>'
              f'<|start_header_id|>user<|end_header_id|>\n{question}<|eot_id|>'
              f'<|start_header_id|>assistant<|end_header_id|>\n')
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split('assistant')[-1].strip()

print(ask('What does the Nepal constitution say about fundamental rights?'))
print('---')
print(ask('भ्रष्टाचारको सजाय के हो?'))